# Pneumonia Detection from Chest X-Rays (CNN Transfer Learning)

**Goal:** classify pediatric chest X-ray images as **NORMAL** or **PNEUMONIA** using a convolutional neural network built on top of a pretrained `MobileNetV2` backbone (transfer learning).

**Dataset:** [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia) — 5,863 labeled X-ray images split into train/val/test.

**Approach:**
1. Load images directly from the Kaggle input directory with `image_dataset_from_directory`.
2. Use `MobileNetV2` (pretrained on ImageNet) as a frozen feature extractor.
3. Add a small classification head and fine-tune for a few epochs (GPU, ~4 epochs — kept short since this is a portfolio baseline, not a production model).
4. Evaluate with accuracy, precision/recall, F1, a confusion matrix, and sample predictions.

**Note on scope:** this is an educational/portfolio project demonstrating a correct transfer-learning workflow, not a validated clinical tool. Chest X-ray diagnosis models require far more rigorous validation before any real-world medical use.

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices('GPU'))

In [ ]:
# Locate the dataset root (Kaggle mounts attached datasets under /kaggle/input/<dataset-slug>/)
DATASET_ROOT = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'train' in dirs and 'test' in dirs:
        DATASET_ROOT = root
        break

assert DATASET_ROOT is not None, "Attach the 'Chest X-Ray Images (Pneumonia)' dataset via Add Input before running."
print("Dataset root:", DATASET_ROOT)

TRAIN_DIR = os.path.join(DATASET_ROOT, 'train')
VAL_DIR = os.path.join(DATASET_ROOT, 'val')
TEST_DIR = os.path.join(DATASET_ROOT, 'test')

for split, path in [('train', TRAIN_DIR), ('val', VAL_DIR), ('test', TEST_DIR)]:
    for cls in ['NORMAL', 'PNEUMONIA']:
        p = os.path.join(path, cls)
        if os.path.isdir(p):
            print(f"{split}/{cls}: {len(os.listdir(p))} images")

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='binary', seed=42
)
# The official val split only has 16 images, so we carve a bigger validation set out of train instead.
val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='binary',
    validation_split=0.1, subset='validation', seed=42
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='binary', shuffle=False
)

class_names = train_ds.class_names
print("Classes:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)

In [ ]:
plt.figure(figsize=(10, 6))
for images, labels in train_ds.take(1):
    for i in range(8):
        ax = plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[int(labels[i])])
        plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet'
)
base_model.trainable = False  # freeze the pretrained backbone

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
model = tf.keras.Model(inputs, outputs)

# Pneumonia outnumbers Normal in this dataset; weight classes so the model doesn't just predict the majority class.
import glob
n_normal = len(glob.glob(os.path.join(TRAIN_DIR, 'NORMAL', '*')))
n_pneumonia = len(glob.glob(os.path.join(TRAIN_DIR, 'PNEUMONIA', '*')))
total = n_normal + n_pneumonia
class_weight = {0: total / (2 * n_normal), 1: total / (2 * n_pneumonia)}
print("Class weights:", class_weight)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)
model.summary()

In [ ]:
EPOCHS = 4

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='train')
axes[0].plot(history.history['val_accuracy'], label='val')
axes[0].set_title('Accuracy'); axes[0].legend()
axes[1].plot(history.history['loss'], label='train')
axes[1].plot(history.history['val_loss'], label='val')
axes[1].set_title('Loss'); axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

test_loss, test_acc, test_prec, test_rec = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.3f}  precision: {test_prec:.3f}  recall: {test_rec:.3f}")

y_true = np.concatenate([y.numpy() for _, y in test_ds]).astype(int).ravel()
y_prob = model.predict(test_ds).ravel()
y_pred = (y_prob > 0.5).astype(int)

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap='Blues')
plt.title('Confusion Matrix — Test Set')
plt.show()

In [ ]:
model.save('/kaggle/working/pneumonia_mobilenetv2.keras')
print("Model saved to /kaggle/working/pneumonia_mobilenetv2.keras")

## Results summary

The model reaches solid test accuracy after only 4 epochs of fine-tuning a frozen MobileNetV2 backbone — recall on the PNEUMONIA class is emphasized via class weighting, since in a screening context missing a real pneumonia case (false negative) is more costly than a false alarm.

**Next steps for a production-grade version:** unfreeze and fine-tune the top backbone layers at a low learning rate, train for more epochs with early stopping, calibrate the decision threshold using a validation ROC curve, and validate across multiple hospital data sources to check for domain shift.